In [6]:
import requests
import urllib.parse
import time
from typing import Dict, List, Optional, Union, Any
from datetime import datetime, timedelta
import json

class DanishParliamentAPI:
    """
    Production-ready client for Danish Parliament Open Data API (oda.ft.dk)

    Features:
    - Comprehensive error handling
    - Automatic retry with exponential backoff
    - Built-in pagination support
    - Rate limiting protection
    - Complete type hints
    """

    def __init__(self, timeout: int = 30, retry_attempts: int = 3):
        """
        Initialize the API client.

        Args:
            timeout: Request timeout in seconds
            retry_attempts: Number of retry attempts for failed requests
        """
        self.base_url = "https://oda.ft.dk/api/"
        self.timeout = timeout
        self.retry_attempts = retry_attempts
        self.last_request_time = 0
        self.min_request_interval = 0.1  # Minimum 100ms between requests

    def _rate_limit(self) -> None:
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()

    def _make_request(self, url: str) -> Dict[str, Any]:
        """
        Make HTTP request with retry logic and error handling.

        Args:
            url: Complete URL to request

        Returns:
            Parsed JSON response

        Raises:
            APIError: For various API errors
            NetworkError: For network-related errors
        """
        self._rate_limit()

        for attempt in range(self.retry_attempts):
            try:
                response = requests.get(url, timeout=self.timeout)

                # Handle different HTTP status codes
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 400:
                    raise APIError(
                        f"Invalid query parameters. Check $expand and $filter syntax. "
                        f"URL: {url}"
                    )
                elif response.status_code == 404:
                    if 'api/' in url and url.count('/') == 4:  # Entity not found
                        raise EntityNotFoundError(f"Entity not found: {url}")
                    else:  # Invalid ID
                        raise RecordNotFoundError(f"Record not found: {url}")
                elif response.status_code == 501:
                    raise UnsupportedOperationError(
                        "Write operations are not supported by this API"
                    )
                else:
                    response.raise_for_status()

            except requests.exceptions.Timeout:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1  # Exponential backoff
                    time.sleep(wait_time)
                    continue
                raise NetworkError(f"Request timed out after {self.timeout} seconds")

            except requests.exceptions.ConnectionError:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1
                    time.sleep(wait_time)
                    continue
                raise NetworkError("Connection error - check your internet connection")

            except requests.exceptions.RequestException as e:
                raise NetworkError(f"Request failed: {str(e)}")

    def _build_url(self, entity: str, **params) -> str:
        """
        Build properly encoded URL with OData parameters.

        Args:
            entity: Entity name (e.g., 'Sag', 'Aktør')
            **params: OData parameters

        Returns:
            Complete URL with encoded parameters
        """
        # Start with base URL and entity
        url = f"{self.base_url}{entity}"

        if not params:
            return url

        # Build query parameters with proper encoding
        query_parts = []
        for key, value in params.items():
            if value is not None:
                # Ensure $ parameters are properly encoded
                if key.startswith('$'):
                    encoded_key = urllib.parse.quote(key, safe='$')
                else:
                    encoded_key = key

                encoded_value = urllib.parse.quote(str(value), safe="(),'/%")
                query_parts.append(f"{encoded_key}={encoded_value}")

        return f"{url}?{'&'.join(query_parts)}"

    def get_cases(self, top: int = 100, skip: int = 0, filter_expr: Optional[str] = None, 
                  expand: Optional[str] = None, select: Optional[str] = None,
                  orderby: Optional[str] = None) -> Dict[str, Any]:
        """
        Get parliamentary cases (Sag) with optional filtering and expansion.

        Args:
            top: Number of records to return (max 100)
            skip: Number of records to skip for pagination
            filter_expr: OData filter expression
            expand: Related entities to include
            select: Specific fields to return
            orderby: Sort order

        Returns:
            API response with case data

        Example:
            # Get recent climate legislation
            cases = api.get_cases(
                filter_expr="substringof('klima', titel)",
                expand="Sagskategori",
                top=50
            )
        """
        params = {'$top': min(top, 100), '$skip': skip}  # Enforce 100 record limit

        if filter_expr:
            params['$filter'] = filter_expr
        if expand:
            params['$expand'] = expand
        if select:
            params['$select'] = select
        if orderby:
            params['$orderby'] = orderby

        url = self._build_url('Sag', **params)
        return self._make_request(url)

    def get_actors(self, top: int = 100, skip: int = 0, filter_expr: Optional[str] = None,
                   expand: Optional[str] = None) -> Dict[str, Any]:
        """
        Get parliamentary actors (Aktør) - politicians, committees, ministries.

        Args:
            top: Number of records to return (max 100)
            skip: Number of records to skip for pagination
            filter_expr: OData filter expression
            expand: Related entities to include

        Returns:
            API response with actor data

        Example:
            # Find all politicians with 'Jensen' in name
            actors = api.get_actors(
                filter_expr="substringof('Jensen', navn)"
            )
        """
        params = {'$top': min(top, 100), '$skip': skip}

        if filter_expr:
            params['$filter'] = filter_expr
        if expand:
            params['$expand'] = expand

        url = self._build_url('Aktør', **params)
        return self._make_request(url)

    def get_voting_records(self, politician_name: str, limit: int = 1000) -> List[Dict[str, Any]]:
        """
        Get all voting records for a specific politician.

        Args:
            politician_name: Full name of politician
            limit: Maximum number of votes to return

        Returns:
            List of voting records with expanded details

        Example:
            votes = api.get_voting_records("Frank Aaen")
        """
        all_votes = []
        skip = 0
        batch_size = 100

        while len(all_votes) < limit and skip < 10000:  # Safety limit
            params = {
                '$expand': 'Afstemning,Aktør',
                '$filter': f"Aktør/navn eq '{politician_name}'",
                '$top': batch_size,
                '$skip': skip
            }

            url = self._build_url('Stemme', **params)
            response = self._make_request(url)

            votes = response.get('value', [])
            if not votes:
                break

            all_votes.extend(votes)
            skip += batch_size

        return all_votes[:limit]

    def get_recent_changes(self, entity: str = 'Sag', hours_back: int = 24) -> Dict[str, Any]:
        """
        Get recent changes to parliamentary data.

        Args:
            entity: Entity to check ('Sag', 'Aktør', 'Afstemning', etc.)
            hours_back: How many hours back to check

        Returns:
            Recent changes in the specified entity

        Example:
            # Check for cases updated in last 4 hours
            recent = api.get_recent_changes('Sag', hours_back=4)
        """
        cutoff_time = datetime.now() - timedelta(hours=hours_back)
        iso_time = cutoff_time.strftime('%Y-%m-%dT%H:%M:%S')

        params = {
            '$filter': f"opdateringsdato gt datetime'{iso_time}'",
            '$orderby': 'opdateringsdato desc',
            '$top': 100
        }

        url = self._build_url(entity, **params)
        return self._make_request(url)

    def get_voting_session_details(self, voting_id: int, expand_votes: bool = True) -> Dict[str, Any]:
        """
        Get detailed information about a voting session.

        Args:
            voting_id: ID of the voting session (Afstemning)
            expand_votes: Whether to include individual vote details

        Returns:
            Voting session with optional vote details
        """
        expand_parts = ['Møde']
        if expand_votes:
            expand_parts.append('Stemme/Aktør')

        params = {
            '$filter': f'id eq {voting_id}',
            '$expand': ','.join(expand_parts)
        }

        url = self._build_url('Afstemning', **params)
        response = self._make_request(url)

        if response.get('value'):
            return response['value'][0]
        else:
            raise RecordNotFoundError(f"Voting session {voting_id} not found")

    def search_documents(self, search_term: str, include_files: bool = False) -> Dict[str, Any]:
        """
        Search parliamentary documents by title.

        Args:
            search_term: Term to search for in document titles
            include_files: Whether to include file download URLs

        Returns:
            Matching documents
        """
        params = {
            '$filter': f"substringof('{search_term}', titel)",
            '$top': 100
        }

        if include_files:
            params['$expand'] = 'Fil'

        url = self._build_url('Dokument', **params)
        return self._make_request(url)

    def get_entity_count(self, entity: str) -> int:
        """
        Get total count of records in an entity.

        Args:
            entity: Entity name

        Returns:
            Total number of records
        """
        params = {
            '$inlinecount': 'allpages',
            '$top': 1
        }

        url = self._build_url(entity, **params)
        response = self._make_request(url)

        count_str = response.get('odata.count', '0')
        return int(count_str)


# Custom Exception Classes
class APIError(Exception):
    """Base exception for API errors."""
    pass

class NetworkError(APIError):
    """Network-related errors."""
    pass

class EntityNotFoundError(APIError):
    """Entity does not exist."""
    pass

class RecordNotFoundError(APIError):
    """Specific record does not exist."""
    pass

class UnsupportedOperationError(APIError):
    """Operation not supported by API."""
    pass

In [ ]:
import pandas as pd
api = DanishParliamentAPI()
# Query all votings from Afstemning with pagination
all_voting_data = []
skip = 0
batch_size = 100

while True:
    votings_response = api._make_request(api._build_url('Afstemning', **{
        '$top': batch_size, 
        '$expand': 'Møde',
        '$skip': skip
    }))
    
    voting_data = votings_response.get('value', [])
    if not voting_data:
        break
    
    all_voting_data.extend(voting_data)
    skip += batch_size

votings_df = pd.DataFrame(all_voting_data)
print(f"Total votings: {len(votings_df)}")
print(votings_df.head())

NameError: name 'pd' is not defined

In [16]:
votings_df = pd.json_normalize(all_voting_data)

In [ ]:
votings_df.to_csv('votings.csv', index=False)
print("votings_df saved to 'votings.csv'")

In [ ]:
votings_df

In [17]:
# Create a DataFrame from the parliamentary periods data
periods_data = [
    ("1", "4. december 1849 – 4. august 1852", 101, 974),
    ("2", "4. august 1852 – 26. februar 1853", 101, 206),
    ("3", "26. februar 1853 – 27. maj 1853", 101, 90),
    ("4", "27. maj 1853 – 1. december 1854", 101, 553),
    ("5", "1. december 1854 – 14. juni 1855", 101, 195),
    ("6", "14. juni 1855 – 14. juni 1858", 101, 1096),
    ("7", "14. juni 1858 – 14. juni 1861", 101, 1096),
    ("8", "14. juni 1861 – 7. juni 1864", 101, 1089),
    ("9", "7. juni 1864 – 4. juni 1866", 101, 727),
    ("10", "4. juni 1866 – 12. oktober 1866", 102, 130),
    ("11", "12. oktober 1866 – 22. september 1869", 102, 1076),
    ("12", "22. september 1869 – 20. september 1872", 102, 1094),
    ("13", "20. september 1872 – 14. november 1873", 102, 420),
    ("14", "14. november 1873 – 25. april 1876", 102, 893),
    ("15", "25. april 1876 – 3. januar 1879", 102, 983),
    ("16", "3. januar 1879 – 24. maj 1881", 102, 872),
    ("17", "24. maj 1881 – 26. juli 1881", 102, 63),
    ("18", "26. juli 1881 – 25. juni 1884", 102, 1065),
    ("19", "25. juni 1884 – 28. januar 1887", 102, 947),
    ("20", "28. januar 1887 – 21. januar 1890", 102, 1089),
    ("21", "21. januar 1890 – 20. april 1892", 102, 820),
    ("22", "20. april 1892 – 9. april 1895", 102, 1084),
    ("23", "9. april 1895 – 5. april 1898", 114, 1092),
    ("24", "5. april 1898 – 3. april 1901", 114, 1093),
    ("25", "3. april 1901 – 16. juni 1903", 114, 804),
    ("26", "16. juni 1903 – 29. maj 1906", 114, 1078),
    ("27", "29. maj 1906 – 25. maj 1909", 114, 1092),
    ("28", "25. maj 1909 – 20. maj 1910", 114, 360),
    ("29", "20. maj 1910 – 20. maj 1913", 114, 1096),
    ("30", "20. maj 1913 – 7. maj 1915", 114, 717),
    ("31", "7. maj 1915 – 22. april 1918", 114, 1081),
    ("32", "22. april 1918 – 26. april 1920", 140, 735),
    ("33", "26. april 1920 – 6. juli 1920", 140, 71),
    ("34", "6. juli 1920 – 21. september 1920", 140, 77),
    ("35", "21. september 1920 – 11. april 1924", 149, 1298),
    ("36", "11. april 1924 – 2. december 1926", 149, 965),
    ("37", "2. december 1926 – 24. april 1929", 149, 874),
    ("38", "24. april 1929 – 16. november 1932", 149, 1302),
    ("39", "16. november 1932 – 22. oktober 1935", 149, 1070),
    ("40", "22. oktober 1935 – 3. april 1939", 149, 1259),
    ("41", "3. april 1939 – 23. marts 1943", 149, 1450),
    ("42", "23. marts 1943 – 30. oktober 1945", 149, 952),
    ("43", "30. oktober 1945 – 28. oktober 1947", 149, 728),
    ("44", "28. oktober 1947 – 5. september 1950", 150, 1043),
    ("45", "5. september 1950 – 21. april 1953", 151, 959),
    ("46", "21. april 1953 – 22. september 1953", 151, 154),
    ("47", "22. september 1953 – 14. maj 1957", 179, 1330),
    ("48", "14. maj 1957 – 15. november 1960", 179, 1281),
    ("49", "15. november 1960 – 22. september 1964", 179, 1407),
    ("50", "22. september 1964 – 22. november 1966", 179, 791),
    ("51", "22. november 1966 – 23. januar 1968", 179, 427),
    ("52", "23. januar 1968 – 21. september 1971", 179, 1337),
    ("53", "21. september 1971 – 4. december 1973", 179, 805),
    ("54", "4. december 1973 – 9. januar 1975", 179, 401),
    ("55", "9. januar 1975 – 15. februar 1977", 179, 768),
    ("56", "15. februar 1977 – 23. oktober 1979", 179, 980),
    ("57", "23. oktober 1979 – 8. december 1981", 179, 777),
    ("58", "8. december 1981 – 10. januar 1984", 179, 763),
    ("59", "10. januar 1984 – 8. september 1987", 179, 1337),
    ("60", "8. september 1987 – 10. maj 1988", 179, 245),
    ("61", "10. maj 1988 – 12. december 1990", 179, 946),
    ("62", "12. december 1990 – 21. september 1994", 179, 1379),
    ("63", "21. september 1994 – 11. marts 1998", 179, 1267),
    ("64", "11. marts 1998 – 20. november 2001", 179, 1350),
    ("65", "20. november 2001 – 8. februar 2005", 179, 1176),
    ("66", "8. februar 2005 – 13. november 2007", 179, 1008),
    ("67", "13. november 2007 – 15. september 2011", 179, 1402),
    ("68", "15. september 2011 – 18. juni 2015", 179, 1372),
    ("69", "18. juni 2015 – 5. juni 2019", 179, 1448),
    ("70", "5. juni 2019 – 1. november 2022", 179, 1232),
    ("71", "1. november 2022 – nu", 179, None),
]

periods_df = pd.DataFrame(
    periods_data,
    columns=["Period", "Date Range", "Parliamentarians", "Days"]
)
periods_df

,Period,Date Range,Parliamentarians,Days
0,1,4. december 1849 – 4. august 1852,101,974.0
1,2,4. august 1852 – 26. februar 1853,101,206.0
2,3,26. februar 1853 – 27. maj 1853,101,90.0
3,4,27. maj 1853 – 1. december 1854,101,553.0
4,5,1. december 1854 – 14. juni 1855,101,195.0
...,...,...,...,...
66,67,13. november 2007 – 15. september 2011,179,1402.0
67,68,15. september 2011 – 18. juni 2015,179,1372.0
68,69,18. juni 2015 – 5. juni 2019,179,1448.0
69,70,5. juni 2019 – 1. november 2022,179,1232.0


In [18]:
# Convert Møde.dato to datetime
votings_df['Møde.dato'] = pd.to_datetime(votings_df['Møde.dato'])

# Convert periods_df date range to start and end dates
def parse_date_range(date_range):
    dates = date_range.split(' – ')
    return dates[0].strip(), dates[1].strip()

# Create a mapping of periods with their date ranges
periods_df['start_date'] = periods_df['Date Range'].apply(lambda x: parse_date_range(x)[0])
periods_df['end_date'] = periods_df['Date Range'].apply(lambda x: parse_date_range(x)[1])

# Convert to datetime, handling "nu" (now) specially
periods_df['start_date'] = pd.to_datetime(periods_df['start_date'], format='%d. %B %Y', errors='coerce')
periods_df['end_date'] = periods_df['end_date'].apply(
    lambda x: pd.Timestamp.now() if x == 'nu' else pd.to_datetime(x, format='%d. %B %Y', errors='coerce')
)

# Function to find the period for a given date
def find_period(vote_date):
    for _, period in periods_df.iterrows():
        if period['start_date'] <= vote_date <= period['end_date']:
            return period['Period']
    return None

# Add period column
votings_df['Period'] = votings_df['Møde.dato'].apply(find_period)

In [19]:
# Filter votings_df for only the last period (71)
votings_df_period_71 = votings_df[votings_df['Period'] == '71']

print(f"Votings in period 71: {len(votings_df_period_71)}")
print(votings_df_period_71.head())

Votings in period 71: 1379
        id  nummer                                         konklusion  \
8925  9002       1  Forslaget blev vedtaget. For stemte 112 (S, V,...   
8926  9003       4  Forslaget blev vedtaget. For stemte 104 (S, V,...   
8927  9004       6  Forslaget blev vedtaget. For stemte 109 (S, V,...   
8928  9005       9  Forslaget blev forkastet. For stemte 21 (SF, E...   
8929  9006      10  Forslaget blev vedtaget. For stemte 108 (S, V,...   

      vedtaget kommentar  mødeid  typeid  sagstrinid          opdateringsdato  \
8925      True      None   12925       1    240777.0  2022-11-16T13:09:09.243   
8926      True      None   12942       1    240797.0   2022-11-22T13:12:23.95   
8927      True      None   12968       1    240842.0   2022-12-13T13:01:44.23   
8928     False      None   12975       1    240789.0    2022-12-22T09:33:33.1   
8929      True      None   12975       1    241082.0  2022-12-22T09:35:58.123   

      Møde.id  ... Møde.nummer Møde.dagsordenur

In [ ]:
votings_df_period_71